In [1]:
import csv
import re

files = [
    "ansiedad.txt",
    "atencion_paliativa.txt",
    "cuidados_paliativos_pediatria.txt",
    "diabetes.txt",
    "manejo_ictus.txt",
    "prevencion_secundaria_ictus.txt"
]

# ---------- patterns ----------
section_pattern = re.compile(r'^(\d+(?:\.\d+)+)\.')
pregunta_pattern = re.compile(r'^[a-z]\)')
subpregunta_pattern = re.compile(r'^[a-z]\.\d+\.')
contexto_pattern = re.compile(r'^Contexto', re.IGNORECASE)

juicio_pattern = re.compile(r'Juicio:\s*', re.IGNORECASE)
evidencia_pattern = re.compile(r'Evidencia procedente de la investigación:\s*', re.IGNORECASE)
consideraciones_pattern = re.compile(r'Consideraciones adicionales:\s*', re.IGNORECASE)

tema_start_pattern = re.compile(
    r'^(Pregunta|Pregunta para responder|Pregunta a responder|Subpregunta)',
    re.IGNORECASE
)

# patrón para eliminar filas con "ver apartado(s) anterior(es)"
skip_pattern = re.compile(r"ver apartado(s)? anterior(es)?", re.IGNORECASE)


def clean_text(t):
    t = t.strip()

    t = re.sub(r'^[\•\●\-\*]+\s*', '', t)
    t = re.sub(r'^\$?\\bullet\$?\s*', '', t)
    t = re.sub(r'(Juicio:\s*)+', 'Juicio: ', t)

    return t.strip()


rows = []

for filename in files:

    with open(filename, encoding="utf8") as f:
        texts = [clean_text(line) for line in f if line.strip()]

    current_section = ""
    current_tema = ""
    current_pregunta = ""
    current_subpregunta = ""

    juicio = []
    evidencia = []
    consideraciones = []

    mode = None
    collecting_tema = False
    tema_buffer = []

    for text in texts:

        # -------- sección --------
        sec = section_pattern.match(text)
        if sec:
            current_section = sec.group(1)
            continue

        # -------- tema start --------
        if tema_start_pattern.match(text):
            collecting_tema = True
            tema_buffer = []
            continue

        # -------- tema end --------
        if collecting_tema and contexto_pattern.match(text):
            collecting_tema = False
            current_tema = " ".join(tema_buffer).strip()
            continue

        if collecting_tema:
            tema_buffer.append(text)
            continue

        # -------- pregunta --------
        if pregunta_pattern.match(text):

            if juicio or evidencia or consideraciones:
                evidencia_text = " ".join(evidencia)
                if not skip_pattern.search(evidencia_text):
                    rows.append({
                        "archivo": filename,
                        "sección": current_section,
                        "tema": current_tema,
                        "pregunta": current_pregunta,
                        "subpregunta": current_subpregunta,
                        "juicio": " ".join(juicio),
                        "evidencia procedente de la investigación": evidencia_text,
                        "consideraciones adicionales": " ".join(consideraciones)
                    })

            current_pregunta = text
            current_subpregunta = ""

            juicio = []
            evidencia = []
            consideraciones = []
            mode = None

            continue

        # -------- subpregunta --------
        if subpregunta_pattern.match(text):

            if juicio or evidencia or consideraciones:
                evidencia_text = " ".join(evidencia)
                if not skip_pattern.search(evidencia_text):
                    rows.append({
                        "archivo": filename,
                        "sección": current_section,
                        "tema": current_tema,
                        "pregunta": current_pregunta,
                        "subpregunta": current_subpregunta,
                        "juicio": " ".join(juicio),
                        "evidencia procedente de la investigación": evidencia_text,
                        "consideraciones adicionales": " ".join(consideraciones)
                    })

            current_subpregunta = text

            juicio = []
            evidencia = []
            consideraciones = []
            mode = None

            continue

        # -------- juicio --------
        if juicio_pattern.search(text):
            mode = "juicio"
            text = juicio_pattern.split(text, 1)[1]
            juicio.append(text)
            continue

        # -------- evidencia --------
        if evidencia_pattern.search(text):
            mode = "evidencia"
            text = evidencia_pattern.split(text, 1)[1]
            evidencia.append(text)
            continue

        # -------- consideraciones --------
        if consideraciones_pattern.search(text):
            mode = "consideraciones"
            text = consideraciones_pattern.split(text, 1)[1]
            consideraciones.append(text)
            continue

        # -------- accumulate paragraphs --------
        if mode == "juicio":
            juicio.append(text)

        elif mode == "evidencia":
            evidencia.append(text)

        elif mode == "consideraciones":
            consideraciones.append(text)

    # save last row
    if juicio or evidencia or consideraciones:
        evidencia_text = " ".join(evidencia)
        if not skip_pattern.search(evidencia_text):
            rows.append({
                "archivo": filename,
                "sección": current_section,
                "tema": current_tema,
                "pregunta": current_pregunta,
                "subpregunta": current_subpregunta,
                "juicio": " ".join(juicio),
                "evidencia procedente de la investigación": evidencia_text,
                "consideraciones adicionales": " ".join(consideraciones)
            })


# ---------- export ----------
with open("dataset.csv", "w", newline="", encoding="utf8") as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "archivo",
            "sección",
            "tema",
            "pregunta",
            "subpregunta",
            "juicio",
            "evidencia procedente de la investigación",
            "consideraciones adicionales"
        ]
    )

    writer.writeheader()
    writer.writerows(rows)

print("CSV created successfully")

CSV created successfully
